# using keras-tuner

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
import joblib
import os

# --- 導入 TensorFlow 和 Keras ---
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.utils import to_categorical
    # 引入 Keras Tuner
    import kerastuner as kt
    print("\nTensorFlow, Keras, Keras Tuner 載入成功。")
except ImportError as e:
    print(f"\nImport Error: {e}. 請確保已安裝 TensorFlow, Keras, 和 Keras Tuner。")
    print("執行 'pip install tensorflow keras-tuner'")
    exit() # 如果 TensorFlow 沒安裝，則終止程式

# --- 資料載入與預處理 (沿用你的程式碼) ---
try:
    df = pd.read_csv('./Data/278k_song_labelled.csv')
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    target = 'labels'
    if target not in df.columns:
        raise ValueError(f"Label column '{target}' not found in the DataFrame.")
    X = df.drop(target, axis=1)
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_input = X_train_scaled
    X_test_input = X_test_scaled
    print("成功載入並處理了實際數據。")
except FileNotFoundError:
    print("警告: 找不到 './Data/278k_song_labelled.csv'。將使用模擬數據。")
    from sklearn.datasets import make_classification
    n_samples = 500
    n_features = 11
    n_classes = 4
    X_sim, y_sim = make_classification(n_samples=n_samples, n_features=n_features, n_informative=5, n_redundant=2, n_classes=n_classes, n_clusters_per_class=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X_sim, y_sim, test_size=0.2, random_state=42, stratify=y_sim)
    scaler = StandardScaler()
    X_train_input = scaler.fit_transform(X_train)
    X_test_input = scaler.transform(X_test)
    print("已創建模擬數據。")
except ValueError as ve:
    print(f"數據處理錯誤: {ve}")
    exit()

# 確保輸入是 NumPy 陣列
X_train_np = np.array(X_train_input)
y_train_np = np.array(y_train)
X_test_np = np.array(X_test_input)
y_test_np = np.array(y_test)

# One-Hot Encoding 標籤
num_classes = len(np.unique(y_train))
y_train_one_hot = to_categorical(y_train_np, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test_np, num_classes=num_classes)

# 創建模型儲存目錄
model_dir = 'saved_models' # 使用與之前一致的目錄名
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"已創建模型儲存目錄: '{model_dir}'")

# --- 定義神經網絡模型構建函數 (供 Keras Tuner 使用) ---
def build_model_for_tuning(hp):
    """
    構建一個可供 Keras Tuner 搜索超參數的神經網絡模型。
    hp (HyperParameters): Keras Tuner 提供的超參數物件。
    """
    model = Sequential()
    
    # 第一層: 根據 hp 選擇神經元數量
    units_layer1 = hp.Int('units_layer1', min_value=32, max_value=128, step=32) # 搜尋 32, 64, 96, 128
    model.add(Dense(units_layer1, activation='relu', input_shape=(X_train_np.shape[1],))) # Input shape
    
    # Dropout 率
    dropout_rate = hp.Float('dropout', min_value=0.3, max_value=0.7, step=0.1) # 搜尋 0.3, 0.4, ..., 0.7
    model.add(Dropout(dropout_rate))
    
    # 第二層: 根據 hp 選擇神經元數量
    units_layer2 = hp.Int('units_layer2', min_value=32, max_value=64, step=32) # 搜尋 32, 64
    model.add(Dense(units_layer2, activation='relu'))
    model.add(Dropout(dropout_rate)) # 使用與第一層相同的 dropout 率

    # 輸出層
    model.add(Dense(num_classes, activation='softmax'))

    # 選擇學習率和優化器
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4]) # 搜尋 0.01, 0.001, 0.0001
    optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop']) # 搜尋 adam, rmsprop

    if optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    
    # 編譯模型
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

# --- 設置 Keras Tuner ---
# 選擇調優算法 (例如 RandomSearch, Hyperband)
# RandomSearch 適合初始探索
tuner_nn = kt.RandomSearch(
    build_model_for_tuning,
    objective='val_accuracy', # 尋找驗證集上準確率最高的模型
    max_trials=10,            # 嘗試 10 組不同的超參數組合
    executions_per_trial=1,   # 每個組合訓練幾次 (1 次即可，節省時間)
    directory='keras_tuner_dir', # 儲存調優結果的目錄
    project_name='song_classification_nn', # 項目名稱
    overwrite=True # 如果項目已存在，則覆蓋
)

print("\n開始神經網絡超參數調優...")
# 訓練過程中使用的驗證集 (這裡從 X_train_np, y_train_one_hot 中劃分)
# validation_split=0.2 的效果與直接傳入 validation_data=(X_val, y_val) 類似
# Keras Tuner 會處理驗證集，我們不需要手動設置 validation_split 給 model.fit
# 這裡我們直接傳入 X_train_np 和 y_train_one_hot，因為 tuner.search 會處理驗證集分割
tuner_nn.search(X_train_np, y_train_one_hot,
                epochs=50, # 每個 trial 訓練的 epoch 數，可以適當設置，讓調優過程不至於太長
                batch_size=32,
                validation_split=0.2, # Keras Tuner 內部會處理這個驗證集
                verbose=1)

# 獲取最佳超參數組合
best_hyperparameters = tuner_nn.get_best_hyperparameters(num_trials=1)[0]
print(f"\n最佳超參數組合: {best_hyperparameters.values}")

# --- 使用最佳超參數構建最終模型 ---
# 構建一個模型，並在所有訓練數據上重新訓練
# Keras Tuner 提供了一個方法來獲取最佳模型
best_nn_model = tuner_nn.hypermodel.build(best_hyperparameters)

# 設定回調函數 (Callbacks)
# 儲存最佳模型
nn_model_filepath_tuned = os.path.join(model_dir, 'nn_best_model_tuned.keras') 
checkpoint_tuned = ModelCheckpoint(nn_model_filepath_tuned, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stopping_tuned = EarlyStopping(monitor='val_accuracy', patience=15, mode='max', restore_best_weights=True, verbose=1) # 增加 patience

print("\n開始使用最佳超參數訓練最終神經網絡模型...")
history_tuned = best_nn_model.fit(X_train_np, y_train_one_hot,
                                epochs=200, # 設置較大的 epoch 數，讓 early stopping 起作用
                                batch_size=32,
                                validation_split=0.2, # 再次從訓練集中分出 20% 作為驗證集
                                callbacks=[checkpoint_tuned, early_stopping_tuned],
                                verbose=1)

print(f"最終神經網絡模型已訓練完成。最佳模型已儲存至: {nn_model_filepath_tuned}")

# --- 評估最終神經網絡模型 ---
# 載入最佳模型（如果 restore_best_weights=True，則模型狀態已是最佳）
# 如果需要明確載入，可以使用 keras.models.load_model(nn_model_filepath_tuned)

# 使用測試集進行評估
loss_tuned, accuracy_tuned = best_nn_model.evaluate(X_test_np, y_test_one_hot, verbose=0)

# 進行預測
y_pred_proba_nn_tuned = best_nn_model.predict(X_test_np)
y_pred_nn_tuned = np.argmax(y_pred_proba_nn_tuned, axis=1)

# 計算評估指標
test_accuracy_nn_tuned = accuracy_score(y_test_np, y_pred_nn_tuned)
test_f1_macro_nn_tuned = f1_score(y_test_np, y_pred_nn_tuned, average='macro', zero_division=0)
conf_matrix_nn_tuned = confusion_matrix(y_test_np, y_pred_nn_tuned)
class_report_nn_tuned = classification_report(y_test_np, y_pred_nn_tuned, zero_division=0)


print(f"\n--- 最終神經網絡模型 (調優後) 評估結果 ---")
print(f"測試集 Loss: {loss_tuned:.4f}")
print(f"測試集 Accuracy (from evaluate): {accuracy_tuned:.4f}")
print(f"測試集 Accuracy (from accuracy_score): {test_accuracy_nn_tuned:.4f}")
print(f"測試集 F1-macro: {test_f1_macro_nn_tuned:.4f}")
print(f"混淆矩陣:\n{conf_matrix_nn_tuned}\n")
print(f"分類報告:\n{class_report_nn_tuned}\n")

# 儲存訓練歷史
history_df_tuned = pd.DataFrame(history_tuned.history)
history_df_tuned.to_csv(os.path.join(model_dir, 'nn_training_history_tuned.csv'))
print(f"最終神經網絡訓練歷史已儲存至: {os.path.join(model_dir, 'nn_training_history_tuned.csv')}")

# --- 比較模型表現 ---
print("\n" + "="*40)
print("比較所有模型的表現 (神經網絡已調優)：")
print("="*40)

# 這裡可以整合之前所有模型的結果 (Logistic Regression, LinearSVC, RandomForest, GaussianNB)
# 並與調優後的神經網絡模型進行比較。
# 你需要將調優後的神經網絡模型的結果添加到一個類似 `all_best_models` 的結構中。

# 假設你已經有了其他模型的 best_model_details 資訊
# 例如: all_best_models['Neural Network (Tuned)'] = { ... }

# 為了方便演示，我們在這裡只顯示 NN 的結果。
# 你可以根據需要整合之前的結果。

print("調優後的神經網絡模型表現:")
print(f"  模型名稱: Neural Network (Tuned)")
print(f"  測試集準確率: {test_accuracy_nn_tuned:.4f}")
print(f"  測試集 F1-macro: {test_f1_macro_nn_tuned:.4f}")
print(f"  儲存路徑: {nn_model_filepath_tuned}")
# print(f"  混淆矩陣:\n{conf_matrix_nn_tuned}") # 可以選擇性輸出
# print(f"  分類報告:\n{class_report_nn_tuned}") # 可以選擇性輸出

# optuna

In [ ]:
# --- 導入 Optuna ---
try:
    import optuna
    print("\nOptuna 載入成功。")
except ImportError:
    print("\nImport Error: Optuna 未安裝。請執行 'pip install optuna'")
    exit()

# --- 資料載入與預處理 (與之前相同) ---
# ... (這部分程式碼保持不變) ...

# 確保輸入是 NumPy 陣列
X_train_np = np.array(X_train_input)
y_train_np = np.array(y_train)
X_test_np = np.array(X_test_input)
y_test_np = np.array(y_test)

# One-Hot Encoding 標籤
num_classes = len(np.unique(y_train))
y_train_one_hot = to_categorical(y_train_np, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test_np, num_classes=num_classes)

# 創建模型儲存目錄
model_dir = 'saved_models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"已創建模型儲存目錄: '{model_dir}'")

# --- 定義 Optuna 的目標函數 (Objective Function) ---
# 這個函數會接收一個 `trial` 物件，並返回一個評估指標 (例如驗證準確率)
def objective(trial):
    """
    Optuna 的目標函數，用於構建和評估一個 Keras 模型。
    """
    # 1. 定義超參數空間
    units_layer1 = trial.suggest_int('units_layer1', 32, 128, step=32)
    dropout_rate = trial.suggest_float('dropout', 0.3, 0.7, step=0.1)
    units_layer2 = trial.suggest_int('units_layer2', 32, 64, step=32)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True) # 使用 log scale 搜尋學習率
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop'])

    # 2. 構建 Keras 模型
    model = Sequential()
    model.add(Dense(units_layer1, activation='relu', input_shape=(X_train_np.shape[1],)))
    model.add(Dropout(dropout_rate))
    model.add(Dense(units_layer2, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(num_classes, activation='softmax'))

    # 選擇優化器
    if optimizer_name == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # 3. 設定回調函數 (Callbacks)
    # 這裡我們不直接使用 EarlyStopping，因為 Optuna 的 Trial 運行時間受限
    # 我們會在 objective 函數的返回值中考慮訓練過程的表現
    # 如果需要 EarlyStopping，可以通過 `tf.keras.callbacks.EarlyStopping` 整合，但會讓 trial 執行時間不固定
    
    # 訓練模型
    history = model.fit(X_train_np, y_train_one_hot,
                        epochs=50, # 每個 trial 的 epoch 數，可以根據需要調整
                        batch_size=32,
                        validation_split=0.2, # 使用訓練集的 20% 作為驗證集
                        verbose=0) # verbosity=0 關閉訓練過程的詳細輸出

    # 4. 返回評估指標 (這個值是 Optuna 要優化的目標)
    # 我們通常關注驗證集上的準確率
    validation_accuracy = max(history.history['val_accuracy']) # 獲取驗證準確率的最大值
    
    # 也可以加入其他指標，例如訓練時的最大準確率
    # train_accuracy = max(history.history['accuracy'])
    # return validation_accuracy # Optuna 會嘗試最大化這個值

    # 如果你希望得到的是驗證集的最後一個 epoch 的準確率，可以使用：
    # return history.history['val_accuracy'][-1]

    # 為了讓 Optuna 盡可能穩定，建議返回驗證集的平均準確率或準確率的最大值
    return validation_accuracy

# --- 啟動 Optuna 的超參數搜索 ---
print("\n開始使用 Optuna 進行神經網絡超參數調優...")

# 創建一個 Optuna Study
# direction='maximize' 表示我們要最大化目標函數的值 (準確率)
study = optuna.create_study(direction='maximize',
                            study_name='song_classification_nn_optuna',
                            sampler=optuna.samplers.TPESampler(), # TPE 是一種常用的優化算法
                            storage='sqlite:///optuna_nn_study.db', # 儲存結果到 SQLite 資料庫
                            load_if_exists=True) # 如果資料庫已存在，則載入

# 執行搜索 (這裡設置了 10 個 trials)
# n_trials: 總共嘗試多少組超參數組合
# timeout: 可選，設置最大運行時間 (秒)
study.optimize(objective, n_trials=10, timeout=None) # timeout=60*10 設定為 10 分鐘

print("\nOptuna 超參數調優完成。")

# --- 獲取最佳超參數和最佳模型 ---
best_trial = study.best_trial
best_hyperparams_optuna = best_trial.params
best_validation_accuracy = best_trial.value

print(f"\n最佳驗證準確率: {best_validation_accuracy:.4f}")
print(f"最佳超參數組合 (Optuna): {best_hyperparams_optuna}")

# --- 使用最佳超參數構建最終模型 ---
# 重新構建模型，使用從 Optuna 找到的最佳參數
best_nn_model_optuna = Sequential()
units_layer1 = best_hyperparams_optuna['units_layer1']
dropout_rate = best_hyperparams_optuna['dropout']
units_layer2 = best_hyperparams_optuna['units_layer2']
learning_rate = best_hyperparams_optuna['learning_rate']
optimizer_name = best_hyperparams_optuna['optimizer']

best_nn_model_optuna.add(Dense(units_layer1, activation='relu', input_shape=(X_train_np.shape[1],)))
best_nn_model_optuna.add(Dropout(dropout_rate))
best_nn_model_optuna.add(Dense(units_layer2, activation='relu'))
best_nn_model_optuna.add(Dropout(dropout_rate))
best_nn_model_optuna.add(Dense(num_classes, activation='softmax'))

if optimizer_name == 'adam':
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
elif optimizer_name == 'rmsprop':
    optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

best_nn_model_optuna.compile(optimizer=optimizer,
                             loss='categorical_crossentropy',
                             metrics=['accuracy'])

# --- 在所有訓練數據上重新訓練最終模型 ---
# 設定回調函數
nn_model_filepath_optuna_tuned = os.path.join(model_dir, 'nn_best_model_optuna_tuned.keras')
checkpoint_optuna_tuned = ModelCheckpoint(nn_model_filepath_optuna_tuned, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stopping_optuna_tuned = EarlyStopping(monitor='val_accuracy', patience=20, mode='max', restore_best_weights=True, verbose=1) # 增加 patience

print("\n開始使用最佳超參數和所有訓練數據訓練最終神經網絡模型...")
history_optuna_tuned = best_nn_model_optuna.fit(X_train_np, y_train_one_hot,
                                               epochs=300, # 設置更大的 epoch 數，讓 early stopping 發揮作用
                                               batch_size=32,
                                               validation_split=0.2, # 再次使用 20% 作為驗證集
                                               callbacks=[checkpoint_optuna_tuned, early_stopping_optuna_tuned],
                                               verbose=1)

print(f"最終神經網絡模型 (Optuna 調優後) 已訓練完成。最佳模型已儲存至: {nn_model_filepath_optuna_tuned}")

# --- 評估最終神經網絡模型 (Optuna 調優後) ---
loss_optuna_tuned, accuracy_optuna_tuned = best_nn_model_optuna.evaluate(X_test_np, y_test_one_hot, verbose=0)

y_pred_proba_nn_optuna_tuned = best_nn_model_optuna.predict(X_test_np)
y_pred_nn_optuna_tuned = np.argmax(y_pred_proba_nn_optuna_tuned, axis=1)

test_accuracy_nn_optuna_tuned = accuracy_score(y_test_np, y_pred_nn_optuna_tuned)
test_f1_macro_nn_optuna_tuned = f1_score(y_test_np, y_pred_nn_optuna_tuned, average='macro', zero_division=0)
conf_matrix_nn_optuna_tuned = confusion_matrix(y_test_np, y_pred_nn_optuna_tuned)
class_report_nn_optuna_tuned = classification_report(y_test_np, y_pred_nn_optuna_tuned, zero_division=0)

print(f"\n--- 最終神經網絡模型 (Optuna 調優後) 評估結果 ---")
print(f"測試集 Loss: {loss_optuna_tuned:.4f}")
print(f"測試集 Accuracy (from evaluate): {accuracy_optuna_tuned:.4f}")
print(f"測試集 Accuracy (from accuracy_score): {test_accuracy_nn_optuna_tuned:.4f}")
print(f"測試集 F1-macro: {test_f1_macro_nn_optuna_tuned:.4f}")
print(f"混淆矩陣:\n{conf_matrix_nn_optuna_tuned}\n")
print(f"分類報告:\n{class_report_nn_optuna_tuned}\n")

# 儲存訓練歷史
history_df_optuna_tuned = pd.DataFrame(history_optuna_tuned.history)
history_df_optuna_tuned.to_csv(os.path.join(model_dir, 'nn_training_history_optuna_tuned.csv'))
print(f"最終神經網絡訓練歷史 (Optuna 調優後) 已儲存至: {os.path.join(model_dir, 'nn_training_history_optuna_tuned.csv')}")

# --- 整合和比較所有模型的表現 ---
# 你可以創建一個字典來儲存所有模型的最佳表現，類似於之前的 all_best_models
# 例如：
# all_models_performance = {
#     'Logistic Regression': {...},
#     'Linear SVC': {...},
#     'Random Forest': {...},
#     'Gaussian Naive Bayes': {...},
#     'Neural Network (Optuna Tuned)': {
#         'model': best_nn_model_optuna,
#         'params': best_hyperparams_optuna,
#         'cv_score': best_validation_accuracy, # Optuna's best trial value
#         'test_accuracy': test_accuracy_nn_optuna_tuned,
#         'test_f1_macro': test_f1_macro_nn_optuna_tuned,
#         'confusion_matrix': conf_matrix_nn_optuna_tuned,
#         'classification_report': class_report_nn_optuna_tuned
#     }
# }

# 之後，你可以遍歷 all_models_performance 來找到整體表現最好的模型。